In [6]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

# Parameters
max_features = 10000  # Number of words to consider as features
maxlen = 200          # Cut texts after 200 words

print("Loading data...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
print(f"{len(x_train)} train sequences, {len(x_test)} test sequences")

Loading data...
25000 train sequences, 25000 test sequences


In [7]:
print("Pad sequences (samples x time)")
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

Pad sequences (samples x time)
x_train shape: (25000, 200)
x_test shape: (25000, 200)


In [8]:
model = Sequential([
    # Embedding layer: maps vocab indices to 128-dimensional vectors
    Embedding(max_features, 128, input_length=maxlen),

    # Convolutional layer
    Conv1D(filters=128, kernel_size=5, activation='relu'),

    # Global Max Pooling reduces the output to a single vector representing the whole sequence
    GlobalMaxPooling1D(),

    # Fully connected layers
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Sigmoid for binary classification (Pos/Neg)
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [4]:
batch_size = 32
epochs = 5

history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_data=(x_test, y_test))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 95s 119ms/step - accuracy: 0.7914 - loss: 0.4278 - val_accuracy: 0.8820 - val_loss: 0.2837
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 93s 119ms/step - accuracy: 0.9243 - loss: 0.2052 - val_accuracy: 0.8811 - val_loss: 0.2896
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 139s 115ms/step - accuracy: 0.9753 - loss: 0.0812 - val_accuracy: 0.8834 - val_loss: 0.3341
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 88s 113ms/step - accuracy: 0.9935 - loss: 0.0250 - val_accuracy: 0.8826 - val_loss: 0.4307
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 143s 115ms/step - accuracy: 0.9973 - loss: 0.0112 - val_accuracy: 0.8695 - val_loss: 0.5607


In [5]:
score, acc = model.evaluate(x_test, y_test, batch_size=batch_size)
print(f'Test loss: {score:.4f}')
print(f'Test accuracy: {acc:.4f}')

# Quick logic to test a custom review
def predict_sentiment(text_indices):
    # This assumes text_indices is already preprocessed/padded
    prediction = model.predict(np.array([text_indices]))
    return "Positive" if prediction > 0.5 else "Negative"

print("\nModel is ready for inference!")

782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.8695 - loss: 0.5607
Test loss: 0.5607
Test accuracy: 0.8695

Model is ready for inference!
